# 실험 09 | 최종 모델 후보 선택 및 안정성 검증

**연구 질문**: 환자별 변이 수를 반영한 Interaction 특징은 다른 모델에서도 도움이 되는가? 모델·특징 선택을 검증 데이터에서 분리했을 때 Macro F1 개선은 유지되는가?

- **외부 3-Fold**: 모델 선택의 결과만 평가 (평가 폴드의 정답/오류/변이율을 선택 기준으로 사용하지 않음)
- **내부 2-Fold**: `Binary / Count / Interaction × SGD / XGBoost × 제한된 하이퍼파라미터`를 비교하여 선택
- **동일 이진 유전체 프로파일 그룹 분리**: 중복 환자 *동일인* 가정이 아닌 입력 프로파일 중복 보호
- **테스트 데이터 금지**: Test 로딩·통계 계산·인코더 `fit`·튜닝·선택 일절 하지 않음

> 주의: 이미 실험 01~08에서 전체 Train의 검증 결과를 확인한 상태이므로 이 중첩 CV도 엄밀한 의미에서 완전히 새로운 독립 외부 검증은 아닙니다. **탐색적 재검증**으로 보고, 공식 성능은 공개 Test 점수가 아닌 Train 내부 검증 성능으로 표시하세요.

필요한 파일: `outputs/X_train_binary.npz`, `outputs/y_train.npy` (실험 01에서 생성). 실행 결과는 `experiment_09/`에 저장됩니다.

In [ ]:
# 최초 설치가 필요하면 아래 명령을 VS Code 터미널에서 한 번 실행하세요.
# python -m pip install -U numpy pandas scipy scikit-learn xgboost matplotlib

from pathlib import Path
import hashlib, json, time, warnings
from itertools import product
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from IPython.display import display
from xgboost import XGBClassifier

SEED = 42
QUICK_CHECK = False  # True: 1 outer fold + SGD 후보로 동작 검사. 최종 결과는 반드시 False!
OUTER_SPLITS = 3
INNER_SPLITS = 2
ROOT = Path.cwd()
DATA_DIR = ROOT / 'outputs'
if not (DATA_DIR / 'X_train_binary.npz').exists():
    DATA_DIR = ROOT / 'exp08_smoke' / 'outputs'  # 배포 예시 구조 대응
OUT = ROOT / 'experiment_09'
OUT.mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR.resolve())
assert (DATA_DIR/'X_train_binary.npz').exists() and (DATA_DIR/'y_train.npy').exists(), 'outputs 폴더를 먼저 준비하세요.'


## 1. 입력 데이터 로딩 및 중복 프로파일 그룹

In [ ]:
X = sparse.load_npz(DATA_DIR/'X_train_binary.npz').tocsr().astype(np.float32)
X.sum_duplicates(); X.eliminate_zeros(); X.sort_indices()
y_text = np.load(DATA_DIR/'y_train.npy', allow_pickle=True).astype(str)
assert X.shape == (6201,4384) and len(y_text)==len(X.indptr)-1
assert np.isin(X.data,[0,1]).all()
encoder = LabelEncoder().fit(y_text)  # Train 라벨만 사용
classes = encoder.classes_
y = encoder.transform(y_text)
count = np.diff(X.indptr).astype(int)
profile_hashes = [hashlib.sha256(X.indices[X.indptr[i]:X.indptr[i+1]].tobytes()).digest() for i in range(X.shape[0])]
groups = pd.factorize(np.asarray(profile_hashes, dtype='|S32'))[0]
print('Train:',len(y),'유전자:',X.shape[1],'암종:',len(classes),'프로파일 그룹:',len(np.unique(groups)))
print('0개 변이:',int((count==0).sum()))


## 2. Fold 내부에서만 전처리를 학습하는 특징 변환기

In [ ]:
def make_features(X_fit, X_other, strategy):
    # 모든 fit 작업은 반드시 내부/외부의 '훈련 인덱스'에서만 수행
    if strategy == 'Binary':
        return X_fit, X_other, {'features':int(X_fit.shape[1])}
    count_fit=np.diff(X_fit.indptr).astype(np.float32)
    count_other=np.diff(X_other.indptr).astype(np.float32)
    scaler=StandardScaler()
    fit_c=scaler.fit_transform(np.log1p(count_fit).reshape(-1,1)).astype(np.float32)
    other_c=scaler.transform(np.log1p(count_other).reshape(-1,1)).astype(np.float32)
    fit_parts=[X_fit, sparse.csr_matrix(fit_c)]
    other_parts=[X_other, sparse.csr_matrix(other_c)]
    metadata={'features':int(X_fit.shape[1]+1),'count_mean_train':float(scaler.mean_[0])}
    if strategy=='Interaction':
        q25,q75=np.percentile(count_fit,[25,75])
        for predicate in (lambda a:a<=q25, lambda a:a>=q75):
            fit_parts.append(X_fit.multiply(predicate(count_fit).astype(np.float32)[:,None]).tocsr())
            other_parts.append(X_other.multiply(predicate(count_other).astype(np.float32)[:,None]).tocsr())
        metadata.update({'q25_train':float(q25),'q75_train':float(q75),'features':int(3*X_fit.shape[1]+1)})
    return sparse.hstack(fit_parts,format='csr'),sparse.hstack(other_parts,format='csr'),metadata

# 후보의 수를 제한하여 탐색 중 우연한 과최적화를 줄임
CANDIDATES = [
    {'name':'SGD_A','family':'SGD','strategy':s,'alpha':1e-4} for s in ['Binary','Count','Interaction']
] + [
    {'name':'SGD_B','family':'SGD','strategy':s,'alpha':1e-3} for s in ['Binary','Count','Interaction']
] + [
    {'name':'XGB_A','family':'XGBoost','strategy':s,'n_estimators':200,'max_depth':3,'learning_rate':0.06} for s in ['Binary','Interaction']
] + [
    {'name':'XGB_B','family':'XGBoost','strategy':s,'n_estimators':350,'max_depth':4,'learning_rate':0.05} for s in ['Binary','Interaction']
]
if QUICK_CHECK:
    CANDIDATES=[c for c in CANDIDATES if c['family']=='SGD']

def make_model(c):
    if c['family']=='SGD':
        return SGDClassifier(loss='log_loss', alpha=c['alpha'], class_weight='balanced', max_iter=1500, tol=1e-3, random_state=SEED)
    return XGBClassifier(n_estimators=c['n_estimators'], max_depth=c['max_depth'],
        learning_rate=c['learning_rate'], subsample=0.85, colsample_bytree=0.85,
        objective='multi:softprob', tree_method='hist', reg_lambda=2.0,
        n_jobs=4, random_state=SEED, eval_metric='mlogloss')

print('후보 수:',len(CANDIDATES), 'QUICK_CHECK:',QUICK_CHECK)


## 3. 중첩 교차검증

각 Outer Fold에서 Inner CV 점수가 높은 **단 하나의 후보**를 선택하고, 외부 검증셋에는 그 모델을 한 번만 평가합니다. 같은 그룹이 학습/검증에 동시에 나타나면 중단합니다. XGBoost는 전체 암종이 학습 폴드에 들어 있는지 확인합니다.

**소요 시간**은 PC 사양에 따라 크게 달라집니다. `QUICK_CHECK=True`로 구조 확인 후 정식 실행하세요.

In [ ]:
outer_folds=list(StratifiedGroupKFold(n_splits=OUTER_SPLITS,shuffle=True,random_state=SEED).split(X,y,groups))
if QUICK_CHECK:
    outer_folds=outer_folds[:1]
inner_records=[]; outer_records=[]; prediction_records=[]; selection_records=[]
for outer_no,(outer_tr,outer_va) in enumerate(outer_folds,1):
    assert set(groups[outer_tr]).isdisjoint(set(groups[outer_va]))
    if len(np.unique(y[outer_tr])) != len(classes):
        raise RuntimeError(f'Outer {outer_no} 학습 데이터에서 암종 클래스 누락')
    # Inner split은 바깥 학습 데이터만으로 계산 (중요)
    local_folds=list(StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=SEED+outer_no).split(X[outer_tr],y[outer_tr],groups[outer_tr]))
    fold_candidates=[]
    print(f'\nOuter {outer_no} | train {len(outer_tr)} | valid {len(outer_va)}')
    for c in CANDIDATES:
        scores=[]
        for inner_no,(itr,iva) in enumerate(local_folds,1):
            tr_idx=outer_tr[itr]; va_idx=outer_tr[iva]
            assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
            if len(np.unique(y[tr_idx])) != len(classes):
                raise RuntimeError(f'Outer {outer_no} Inner {inner_no} 클래스 누락. 분할 수/시드를 검토하세요.')
            Xt,Xv,_=make_features(X[tr_idx],X[va_idx],c['strategy'])
            model=make_model(c); model.fit(Xt,y[tr_idx]); p=model.predict(Xv)
            f1=f1_score(y[va_idx],p,labels=np.arange(len(classes)),average='macro',zero_division=0)
            scores.append(float(f1))
            inner_records.append({'outer_fold':outer_no,'inner_fold':inner_no,'candidate':c['name']+'_'+c['strategy'],
                                  'family':c['family'],'strategy':c['strategy'],'macro_f1':float(f1),
                                  'valid_n':len(va_idx)})
        fold_candidates.append((np.mean(scores), c))
    fold_candidates.sort(key=lambda item:item[0],reverse=True)
    best_score,best=fold_candidates[0]
    print('선택:',best['name'],best['strategy'],'inner 평균 Macro F1=',round(best_score,4))
    Xt,Xv,metadata=make_features(X[outer_tr],X[outer_va],best['strategy'])
    model=make_model(best); model.fit(Xt,y[outer_tr]); pred=model.predict(Xv)
    f1=f1_score(y[outer_va],pred,labels=np.arange(len(classes)),average='macro',zero_division=0)
    acc=accuracy_score(y[outer_va],pred)
    print('Outer Macro F1:',round(f1,4),'Accuracy:',round(acc,4))
    outer_records.append({'outer_fold':outer_no,'macro_f1':float(f1),'accuracy':float(acc),'n_valid':len(outer_va),
                          'selected_candidate':best['name']+'_'+best['strategy'],'inner_mean':float(best_score),**metadata})
    selection_records.append({'outer_fold':outer_no,'candidate':best['name'],'family':best['family'],'strategy':best['strategy'],
                              'inner_macro_f1':float(best_score), 'settings':json.dumps(best,ensure_ascii=False)})
    for j,pred_idx in zip(outer_va,pred):
        prediction_records.append({'row_index':int(j),'outer_fold':outer_no,'true':str(y_text[j]),'pred':str(classes[int(pred_idx)]),
                                   'correct':int(int(pred_idx)==int(y[j])),'mutation_count':int(count[j])})

inner_df=pd.DataFrame(inner_records); outer_df=pd.DataFrame(outer_records)
oof=pd.DataFrame(prediction_records); selection_df=pd.DataFrame(selection_records)
display(outer_df)
print('Outer Fold 평균 Macro F1:',round(outer_df.macro_f1.mean(),5))
assert not oof.row_index.duplicated().any()
if not QUICK_CHECK: assert sorted(oof.row_index.tolist())==list(range(X.shape[0]))


## 4. 암종별 F1, 소수 암종, 변이 부담 및 혼동 쌍

이 표는 **결과 해석을 위한 사후 분석**입니다. 여기에서 본 암종별 결과로 후보를 재선택하면 검증셋에 과적합될 수 있습니다. 소수 암종은 전체 Train의 클래스 빈도 하위 6개로 정의해 보고하되, 이는 **실험 전 고정된 연구용 정의**로 간주합니다.

In [ ]:
PAIRS=[('KIPAN','KIRC'),('GBMLGG','LGG'),('BRCA','OV')]
minority=pd.Series(y_text).value_counts().sort_values().index[:6].tolist()
rows=[]
for label in classes:
    gt=(oof.true==label).astype(int); pr=(oof.pred==label).astype(int)
    tp=int(((gt==1)&(pr==1)).sum()); fn=int(((gt==1)&(pr==0)).sum()); fp=int(((gt==0)&(pr==1)).sum())
    rows.append({'cancer':label,'n_valid':int(gt.sum()),'f1':f1_score(gt,pr,zero_division=0),
                 'recall':tp/(tp+fn) if tp+fn else 0,'precision':tp/(tp+fp) if tp+fp else 0,
                 'minority':label in minority})
class_df=pd.DataFrame(rows)
print('소수 6개 암종:',minority)
print('OOF 집계 Macro F1:',round(class_df.f1.mean(),5),'소수 암종 Macro F1:',round(class_df.loc[class_df.minority,'f1'].mean(),5))
display(class_df.sort_values('f1').head(12))
pair_df=pd.DataFrame([{'pair':f'{a} <-> {b}', 'errors':int(((oof.true==a)&(oof.pred==b)).sum()+((oof.true==b)&(oof.pred==a)).sum())} for a,b in PAIRS]);display(pair_df)
oof['burden_group']=pd.cut(oof.mutation_count,[-1,0,7,27,np.inf],labels=['0개','1~7개','8~27개','28개 이상'])
burden_df=oof.groupby('burden_group',observed=True).agg(n=('correct','size'),error_rate=('correct',lambda x:1-x.mean())).reset_index();display(burden_df)


## 5. 전체 Train 기반 최종 후보 선택 **(평가가 아님)**

제출 전 최종 후보를 정하는 단계입니다. 전체 Train으로 **그룹 기반 내부 3-Fold**를 한 번 더 수행해 후보별 Macro F1을 비교하고 1개 모델을 선택합니다. 이후 선택한 후보로 전체 Train을 다시 학습해야 합니다(실험 10). 이 단계의 점수는 **최종 일반화 성능이 아닌 선택용 점수**입니다. Test에는 접근하지 않습니다.

In [ ]:
# 추가 학습이 필요하므로 QUICK_CHECK 모드에서는 생략
final_selection=pd.DataFrame()
if not QUICK_CHECK:
    select_folds=list(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED+101).split(X,y,groups))
    selections=[]
    for c in CANDIDATES:
        scores=[]
        for tr,va in select_folds:
            assert set(groups[tr]).isdisjoint(groups[va])
            assert len(np.unique(y[tr]))==len(classes)
            Xt,Xv,_=make_features(X[tr],X[va],c['strategy'])
            m=make_model(c);m.fit(Xt,y[tr]);p=m.predict(Xv)
            scores.append(f1_score(y[va],p,labels=np.arange(len(classes)),average='macro',zero_division=0))
        selections.append({'candidate':c['name']+'_'+c['strategy'],'family':c['family'],'strategy':c['strategy'],
                           'mean_macro_f1':float(np.mean(scores)),'std_macro_f1':float(np.std(scores,ddof=1)),
                           'settings':json.dumps(c,ensure_ascii=False)})
    final_selection=pd.DataFrame(selections).sort_values('mean_macro_f1',ascending=False)
    print('최종 후보 (전체 Train 내부 선택):')
    display(final_selection.head(10))
    print('실험 10에서 채택할 후보:',final_selection.iloc[0]['candidate'])
else:
    print('QUICK_CHECK=True: 최종 후보 선택 단계를 건너뜁니다.')


## 6. 결과 저장

In [ ]:
inner_df.to_csv(OUT/'inner_cv_metrics.csv',index=False,encoding='utf-8-sig')
outer_df.to_csv(OUT/'outer_cv_metrics.csv',index=False,encoding='utf-8-sig')
selection_df.to_csv(OUT/'outer_selected_models.csv',index=False,encoding='utf-8-sig')
oof.to_csv(OUT/'nested_oof_predictions.csv',index=False,encoding='utf-8-sig')
class_df.to_csv(OUT/'class_f1.csv',index=False,encoding='utf-8-sig')
pair_df.to_csv(OUT/'pair_confusion.csv',index=False,encoding='utf-8-sig')
burden_df.to_csv(OUT/'burden_errors.csv',index=False,encoding='utf-8-sig')
if not final_selection.empty: final_selection.to_csv(OUT/'final_selection_cv.csv',index=False,encoding='utf-8-sig')
config={'seed':SEED,'quick_check':QUICK_CHECK,'outer_splits':OUTER_SPLITS,'inner_splits':INNER_SPLITS,
        'candidates':CANDIDATES,'n_rows':int(X.shape[0]),'minority_labels':minority,'test_used':False,
        'warning':'Exploratory nested CV on previously examined Train. Not a pristine external test.'}
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 완료:',OUT.resolve())
print('experiment_09 폴더를 ZIP으로 압축해서 전달해 주세요.')
